Build Constructors Dimension

1. Read silver constructors table

2. Read gold ref_nationality_region table

3. Join the data from constructors with ref_nationality_region using nationality

4. Select the required columns

5. Write the transformed data to gold din_constructors ta

In [0]:
%run "/Workspace/Users/ganeshgadade157@gmail.com/Projects/Formula-f1/00.common/01.envioment_config"

In [0]:
from pyspark.sql.functions import col

# Table name for gold dim_constructors
table_name = f"{catlog_name}.{gold_schema}.dim_constructors"

# Step 1 - Read silver constructors and gold ref_nationality_region tables
df_constructors  = spark.read.table("dev.silver.constructors")
df_ref_nationality_region = spark.read.table("dev.gold.ref_nationality_region")

# Step 2 - Join constructors with nationality region and select required columns
df_dim_constructors = df_constructors.alias("c").join(
    df_ref_nationality_region.alias("n"),
    col("c.nationality") == col("n.nationality"),
    "left"
).select(
    col("c.constructor_id"),
    col("c.name"),
    col("c.nationality"),
    col("n.region").alias("nationality_region")
)

# Step 3 - Write to gold dim_constructors table in delta format
df_dim_constructors.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

In [0]:
table_names = [
    "dev.gold.dim_constructors",
    "dev.gold.dim_drivers",
    "dev.gold.dim_races",
    "dev.gold.fact_session_results"
]

rows = [spark.sql(f"DESCRIBE DETAIL {t}").select("name", "location").collect()[0] for t in table_names]
display(spark.createDataFrame(rows))